# VietMedBridge — 03: Kiểm tra toàn snapshot, audit và freeze candidate

Chạy sau notebook 02, runtime CPU. Validator kiểm official ID/URL, uniqueness,
foreign keys, source hash, exact offsets và bảo toàn mọi input qua documents/failures.
Dedup reducer đọc tất cả shard được manifest chọn; không xóa official IDs.


In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError(
        "Dùng Python 3.11–3.13. Trong Runtime > Change runtime type, "
        "chọn phiên bản runtime có Python trong khoảng này."
    )
if importlib.util.find_spec("google.colab") is None:
    raise RuntimeError("Notebook này được thiết kế cho Google Colab.")

from google.colab import drive
# Chạy lại cell này sau MỖI runtime mới/restart: package chỉ được cài trong phiên Colab.
drive.mount("/content/drive")

# Giữ cùng DATA_ROOT trong cả bốn notebook.
DATA_ROOT = Path("/content/drive/MyDrive/VietMedBridge/data")
WORK_DIR = Path("/content/vmb_work")  # temp files/spill ở ổ local của Colab
DATA_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

REPO_URL = "https://github.com/Platypus27-coder/VietMedBridge.git"
CODE_REVISION = None  # Có thể đặt full commit SHA; mặc định dùng code_lock đã lưu.
EXPECTED_PIPELINE_API = 2
CHECKOUT = Path("/content/VietMedBridge")
lock_path = DATA_ROOT / "code_lock.json"
lock = json.loads(lock_path.read_text()) if lock_path.exists() else {}
if lock and lock.get("repo_url") != REPO_URL:
    raise ValueError("DATA_ROOT đang khóa vào một repo khác.")
if lock and lock.get("pipeline_api") != EXPECTED_PIPELINE_API and not CODE_REVISION:
    raise RuntimeError(
        "DATA_ROOT đang dùng code cũ. Để nâng lên data-v2, đặt CODE_REVISION='main' "
        "ở cell này rồi chạy lại; dùng tên crawl/build run mới. Raw cũ vẫn được giữ."
    )
reference = CODE_REVISION or lock.get("git_commit") or "main"
if not (CHECKOUT / ".git").exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(CHECKOUT)], check=True)
remote = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"], text=True
).strip()
if remote != REPO_URL:
    raise ValueError("Checkout hiện tại không thuộc repo VietMedBridge.")
subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", reference], check=True)
CODE_COMMIT = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "rev-parse", "FETCH_HEAD"], text=True
).strip()
if "vietmedbridge" in sys.modules and globals().get("_VMB_IMPORTED_COMMIT") != CODE_COMMIT:
    raise RuntimeError("Mã nguồn đổi trong runtime đã import package. Restart session rồi chạy lại notebook.")
subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[notebook]"],
    cwd=CHECKOUT,
    check=True,
)
PACKAGE_SOURCE = CHECKOUT / "src"
if not (PACKAGE_SOURCE / "vietmedbridge" / "__init__.py").is_file():
    raise RuntimeError(
        f"Repo checkout thiếu package source: {PACKAGE_SOURCE}; commit={CODE_COMMIT}. "
        "Kiểm tra lại git fetch/checkout ở output của cell Bootstrap."
    )
sys.path.insert(0, str(PACKAGE_SOURCE))
importlib.invalidate_caches()
from vietmedbridge.artifacts import atomic_json, runtime_versions
from vietmedbridge import PIPELINE_API_VERSION
if PIPELINE_API_VERSION != EXPECTED_PIPELINE_API:
    raise RuntimeError("Notebook và package API không khớp; chọn commit data-v2 và restart session.")
_VMB_IMPORTED_COMMIT = CODE_COMMIT
if not lock or CODE_REVISION:
    atomic_json(lock_path, {"repo_url": REPO_URL, "git_commit": CODE_COMMIT, "pipeline_api": PIPELINE_API_VERSION})
PIPELINE_CONFIG = json.loads((CHECKOUT / "configs/data_pipeline.json").read_text())
atomic_json(DATA_ROOT / "runtime.json", {
    "git_commit": CODE_COMMIT, "python": sys.version, "packages": runtime_versions()
})
print("Code commit:", CODE_COMMIT)
import vietmedbridge
print("Package source:", Path(vietmedbridge.__file__).resolve())
print("Dữ liệu/checkpoint:", DATA_ROOT)


## 1. Chọn build và chạy lại golden với tokenizer đã ghim


In [ ]:
from vietmedbridge.artifacts import read_json, atomic_json
from vietmedbridge.dataset import load_snapshot, parquet_path
from vietmedbridge.chunks import ChunkConfig, load_bge_tokenizer, parent_for_child
from vietmedbridge.golden import run_golden_suite
from vietmedbridge.health import health_report, freeze_candidate

CRAWL_RUN = "stage-a-v3"
BUILD_RUN = "stage-a-data-v3"
BUILD_DIR = DATA_ROOT / "processed" / BUILD_RUN
SNAPSHOT = load_snapshot(DATA_ROOT)
OFFICIAL_LINKS = parquet_path(DATA_ROOT, "links_corpus.parquet")
TOKENIZER, TOKENIZER_SPEC = load_bge_tokenizer(read_json(DATA_ROOT / "tokenizer_lock.json")["revision"])
CHUNK_CONFIG = ChunkConfig(**read_json(BUILD_DIR / "config.json")["chunks"])
GOLDEN = run_golden_suite(CHECKOUT / "tests/golden/cases.json", TOKENIZER, TOKENIZER_SPEC, CHUNK_CONFIG)
atomic_json(DATA_ROOT / "reports/golden_latest.json", GOLDEN)
if not GOLDEN["passed"]:
    raise RuntimeError("Golden fail; chưa được freeze candidate.")


## 2. Health report, global dedup và kiểm tra nguồn bằng HTML


In [ ]:
HEALTH = health_report(BUILD_DIR, official_links=OFFICIAL_LINKS,
                       crawl_dir=DATA_ROOT / "crawl" / CRAWL_RUN,
                       work_dir=WORK_DIR, audit_size=PIPELINE_CONFIG["audit_size"])
print(json.dumps({key: HEALTH[key] for key in ("coverage", "documents", "chunks", "dedup")}, ensure_ascii=False, indent=2))
import pandas as pd
from IPython.display import HTML, display
display(pd.DataFrame(HEALTH["by_domain"]))
display(pd.DataFrame(HEALTH["failure_reasons"]))
display(HTML((BUILD_DIR / HEALTH["files"]["audit"]["path"]).read_text()))
print("Golden real-document candidates (chưa có expected assertions):",
      BUILD_DIR / HEALTH["files"]["golden_candidates"]["path"])


## 3. Freeze snapshot candidate

FROZEN_CANDIDATE là snapshot bất biến để xây index/benchmark. Nó chưa được
PROMOTED: cần retrieval với qrels/reference spans, ngân sách đo thật và human audit.
Corpus query hiện không có gold labels. LOW quality vẫn nằm trong corpus eligible.


In [ ]:
CANDIDATE = freeze_candidate(BUILD_DIR, HEALTH, golden_report=GOLDEN)
print(CANDIDATE["state"], CANDIDATE["candidate_manifest_sha256"])
print(CANDIDATE["promotion"])
print("Canonical aliases:", BUILD_DIR / HEALTH["files"]["canonical_aliases"]["path"])
print("Representation aliases:", BUILD_DIR / HEALTH["files"]["representation_aliases"]["path"])


## 4. Team ghi nhận milestone sau khi kiểm tra mẫu nguồn

Copy golden_candidates.json thành reviewed_golden.json trên Drive. Với mỗi nguồn,
kiểm tra raw/source rồi điền expected_key_snippets, đổi annotation_status thành REVIEWED.
Có thể bổ sung expected_title_contains, expected_language, expected_min_text_length,
expected_chunk_count_range. Không tự lấy text hiện tại làm đáp án rồi coi là human review.
Replay cần 100–500 nguồn; 11 fixture synthetic không thay cho bộ golden này.
Chỉ bật AUDIT_APPROVED sau khi người review đã xem health report/source audit và golden thật pass.
Stage A được duyệt mở đường tới Stage B1 10k. B2/C/full còn cần báo cáo retrieval
thật và budget; không tự đánh dấu pass khi chưa chạy. Chỉ một runtime ghi build/report.


In [ ]:
from vietmedbridge.gates import record_milestone, stage_for_rows
from vietmedbridge.golden import run_reviewed_golden

AUDIT_APPROVED = False
REVIEWER = ""  # tên người trong team đã thực hiện audit
REVIEWED_GOLDEN_PATH = DATA_ROOT / "reports/reviewed_golden.json"
REVIEWED_GOLDEN = None
if REVIEWED_GOLDEN_PATH.exists():
    REVIEWED_GOLDEN = run_reviewed_golden(
        REVIEWED_GOLDEN_PATH, DATA_ROOT / "crawl" / CRAWL_RUN, TOKENIZER, TOKENIZER_SPEC,
        config=CHUNK_CONFIG,
    )
    atomic_json(DATA_ROOT / "reports/reviewed_golden_latest.json", REVIEWED_GOLDEN)
    print("Reviewed golden passed:", REVIEWED_GOLDEN["passed"])
if AUDIT_APPROVED:
    MILESTONE = record_milestone(
        DATA_ROOT, CANDIDATE, stage=stage_for_rows(CANDIDATE["counts"]["input_records"]),
        corpus_sha256=SNAPSHOT["files"]["links_corpus.parquet"]["sha256"],
        reviewer=REVIEWER, approved=True, reviewed_golden_report=REVIEWED_GOLDEN,
    )
    print(MILESTONE)
else:
    print("Candidate đã freeze; milestone human review đang chờ team.")
